# 7.9 — Clustering Countries

**Chapter 7, section 7.4** — the scaling argument at the end of it. The starting point for
**Exercise 7**.

**The question this notebook answers:** what does an unscaled feature do to a distance-based
algorithm?

Sixty-five countries, six columns. Three of them are counts of titles in the local Netflix
catalogue and run into the thousands; three are monthly subscription prices and run from about
two to about twenty-seven. K-means minimizes Euclidean distance, which is a sum of squared
coordinate differences — so a column whose differences are three orders of magnitude larger than
another's contributes a million times as much to every distance.

The chapter states the consequence at line 505: an unscaled feature "effectively becomes the only
feature visible to distance-based algorithms." This notebook measures it.

Source: `Notebooks old/Spark-Example-22-Mllib-Clustering.ipynb`, cells 3–9 and 14–15, modernized
and revised. The original's `plot_clustering` helper needed `seaborn` and `adjustText` and drew a
two-dimensional PCA scatter; it is replaced here by the cluster centroids in the original units,
which is what Exercise 7 (c) asks for and is easier to read in a report.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.9")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. Six columns on two very different scales

In [2]:
import pyspark.sql.functions as F

COLUMNS = ["Total_Library_Size", "Num_TV_Shows", "Num_Movies",
           "Cost_Per_Month_Basic", "Cost_Per_Month_Standard", "Cost_Per_Month_Premium"]

data = spark.read.csv(os.path.join(DATA, "netflix-subscription.csv"),
                      header=True, inferSchema=True).cache()
print("countries:", data.count())
data.select("Country", *COLUMNS).show(5)

countries: 65
+---------+------------------+------------+----------+--------------------+-----------------------+----------------------+
|  Country|Total_Library_Size|Num_TV_Shows|Num_Movies|Cost_Per_Month_Basic|Cost_Per_Month_Standard|Cost_Per_Month_Premium|
+---------+------------------+------------+----------+--------------------+-----------------------+----------------------+
|Argentina|              4760|        3154|      1606|                3.74|                    6.3|                  9.26|
|Australia|              6114|        4050|      2064|                7.84|                  12.12|                 16.39|
|  Austria|              5640|        3779|      1861|                9.03|                  14.67|                 20.32|
|  Belgium|              4990|        3374|      1616|               10.16|                  15.24|                 20.32|
|  Bolivia|              4991|        3155|      1836|                7.99|                  10.99|                 13.99|
+-

In [3]:
stats = {c: data.select(F.min(c), F.max(c), F.stddev(c)).first() for c in COLUMNS}

print(f"{'column':<26}{'min':>10}{'max':>10}{'std dev':>12}")
print("-" * 58)
for c, row in stats.items():
    print(f"{c:<26}{row[0]:>10.2f}{row[1]:>10.2f}{row[2]:>12.2f}")

column                           min       max     std dev
----------------------------------------------------------
Total_Library_Size           2274.00   7325.00      980.32
Num_TV_Shows                 1675.00   5234.00      723.01
Num_Movies                    373.00   2387.00      327.28
Cost_Per_Month_Basic            1.97     12.88        1.94
Cost_Per_Month_Standard         3.00     20.46        2.86
Cost_Per_Month_Premium          4.02     26.96        4.04


In [4]:
# What that does to a squared distance. Two countries at opposite ends of each column:
print(f"{'column':<26}{'range':>10}{'range squared':>18}{'share of distance':>20}")
squares = {c: (stats[c][1] - stats[c][0]) ** 2 for c in COLUMNS}
total = sum(squares.values())
for c in COLUMNS:
    r = stats[c][1] - stats[c][0]
    print(f"{c:<26}{r:>10.2f}{squares[c]:>18,.0f}{squares[c] / total:>19.4%}")

column                         range     range squared   share of distance
Total_Library_Size           5051.00        25,512,601           60.4045%
Num_TV_Shows                 3559.00        12,666,481           29.9896%
Num_Movies                   2014.00         4,056,196            9.6036%
Cost_Per_Month_Basic           10.91               119            0.0003%
Cost_Per_Month_Standard        17.46               305            0.0007%
Cost_Per_Month_Premium         22.94               526            0.0012%


The three price columns together account for a fraction of a percent of the worst-case squared
distance. Whatever the clustering below discovers, it will be discovering it about catalogue size.

---

## 2. K-means on the raw columns

In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

assembler = VectorAssembler(inputCols=COLUMNS, outputCol="features")
featured = assembler.transform(data).cache()

raw_model = KMeans(featuresCol="features", k=3, seed=1).fit(featured)
raw_clusters = raw_model.transform(featured).cache()

print("silhouette:",
      round(ClusteringEvaluator(featuresCol="features").evaluate(raw_clusters), 4))

silhouette: 0.7231


In [6]:
def profile(clusters):
    """Cluster sizes and the mean of every original column, per cluster."""
    return (clusters.groupBy("prediction")
            .agg(F.count("*").alias("n"),
                 *[F.round(F.avg(c), 1).alias(c) for c in COLUMNS])
            .orderBy("prediction"))


profile(raw_clusters).show()

+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+
|prediction|  n|Total_Library_Size|Num_TV_Shows|Num_Movies|Cost_Per_Month_Basic|Cost_Per_Month_Standard|Cost_Per_Month_Premium|
+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+
|         0|  6|            3250.2|      2147.3|    1102.8|                 9.6|                   13.7|                  17.6|
|         1| 20|            6380.6|      4323.9|    2056.7|                 8.3|                   12.0|                  15.4|
|         2| 39|            5085.3|      3317.2|    1768.1|                 8.2|                   11.8|                  15.4|
+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+



In [7]:
# The countries in each cluster -- the original notebook's output.
for i in range(3):
    members = [r["Country"] for r in
               raw_clusters.filter(F.col("prediction") == i)
               .orderBy("Country").select("Country").collect()]
    print(f"cluster {i} ({len(members)}):", ", ".join(members))
    print()

cluster 0 (6): Croatia, Finland, Indonesia, Liechtenstein, Moldova, San Marino

cluster 1 (20): Australia, Bulgaria, Canada, Czechia, Estonia, Gibraltar, Hungary, Iceland, India, Ireland, Latvia, Lithuania, Malaysia, Monaco, New Zealand, Philippines, Singapore, Slovakia, United Kingdom, United States

cluster 2 (39): Argentina, Austria, Belgium, Bolivia, Brazil, Chile, Colombia, Costa Rica, Denmark, Ecuador, France, Germany, Greece, Guatemala, Honduras, Hong Kong, Israel, Italy, Japan, Mexico, Netherlands, Norway, Paraguay, Peru, Poland, Portugal, Romania, Russia, South Africa, South Korea, Spain, Sweden, Switzerland, Taiwan, Thailand, Turkey, Ukraine, Uruguay, Venezuela



---

## 3. What the clustering actually used

A cluster profile is easier to read if each column's spread across the clusters is expressed in
units of that column's own standard deviation. A column the clustering ignored has clusters whose
means are all about the same, so its spread is near zero.

In [8]:
def influence(clusters, label):
    p = profile(clusters).collect()
    print(f"{label}")
    print(f"   {'column':<26}{'lowest mean':>13}{'highest mean':>14}{'spread / sd':>13}")
    for c in COLUMNS:
        means = [r[c] for r in p]
        spread = (max(means) - min(means)) / stats[c][2]
        print(f"   {c:<26}{min(means):>13.1f}{max(means):>14.1f}{spread:>13.2f}")
    print()


influence(raw_clusters, "unscaled")

unscaled
   column                      lowest mean  highest mean  spread / sd
   Total_Library_Size               3250.2        6380.6         3.19
   Num_TV_Shows                     2147.3        4323.9         3.01
   Num_Movies                       1102.8        2056.7         2.91
   Cost_Per_Month_Basic                8.2           9.6         0.72
   Cost_Per_Month_Standard            11.8          13.7         0.66
   Cost_Per_Month_Premium             15.4          17.6         0.54



The three count columns separate the clusters by about three standard deviations each. The three
price columns manage about two-thirds of one — and even that is carried entirely by a single
six-country cluster. Look back at the profile in section 2: the two clusters holding **fifty-nine
of the sixty-five countries** have mean prices that agree to within twenty cents on all three
tiers.

Sixty-five countries have been partitioned by how many titles they can watch. Nothing was said
about what they pay, even though half the input columns were about exactly that.

---

## 4. The same clustering, scaled

`StandardScaler` divides each feature by its standard deviation, so every column contributes on
comparable terms. `withMean=True` also centres them, which is safe here because these vectors are
dense — on the sparse text vectors of notebook 7.8 it would have been a serious mistake.

Note that the scaler is an **estimator**: it has to compute the six standard deviations before it
can divide by them, which is one pass over the data.

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled",
                        withStd=True, withMean=True)

scaled_pipeline = Pipeline(stages=[
    scaler,
    KMeans(featuresCol="scaled", k=3, seed=1),
]).fit(featured)

scaled_clusters = scaled_pipeline.transform(featured).cache()

print("silhouette on the scaled features:",
      round(ClusteringEvaluator(featuresCol="scaled").evaluate(scaled_clusters), 4))

silhouette on the scaled features: 0.4507


In [10]:
profile(scaled_clusters).show()

+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+
|prediction|  n|Total_Library_Size|Num_TV_Shows|Num_Movies|Cost_Per_Month_Basic|Cost_Per_Month_Standard|Cost_Per_Month_Premium|
+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+
|         0| 34|            5923.9|      3992.5|    1931.4|                 8.9|                   12.8|                  16.6|
|         1| 22|            4927.5|      3140.5|    1787.0|                 6.6|                    9.5|                  12.2|
|         2|  9|            3957.8|      2655.1|    1302.7|                10.6|                   15.2|                  20.4|
+----------+---+------------------+------------+----------+--------------------+-----------------------+----------------------+



In [11]:
influence(scaled_clusters, "scaled")

scaled
   column                      lowest mean  highest mean  spread / sd
   Total_Library_Size               3957.8        5923.9         2.01
   Num_TV_Shows                     2655.1        3992.5         1.85
   Num_Movies                       1302.7        1931.4         1.92
   Cost_Per_Month_Basic                6.6          10.6         2.06
   Cost_Per_Month_Standard             9.5          15.2         1.99
   Cost_Per_Month_Premium             12.2          20.4         2.03



Now all six columns separate the clusters by a comparable amount, and the partition means
something different: one group with large catalogues at middling prices, one paying least, and one
paying most for the smallest catalogues. That is a statement about the market. The unscaled
version was a statement about catalogue size with five columns along for the ride.

**The silhouette went down.** From about 0.72 to about 0.45, and that is not a sign that the
scaled clustering is worse. The unscaled silhouette was high because a one-dimensional problem is
easy to separate cleanly; the scaled problem is genuinely six-dimensional and its clusters
genuinely overlap. A metric computed in the feature space is a statement about that feature space,
so two silhouettes computed in different spaces are not comparable.

In [12]:
# The two partitions are not the same, and the labels are arbitrary, so they are compared
# through the contingency table rather than by matching cluster numbers.
(raw_clusters.select("Country", F.col("prediction").alias("unscaled"))
 .join(scaled_clusters.select("Country", F.col("prediction").alias("scaled")), "Country")
 .groupBy("unscaled").pivot("scaled").count().orderBy("unscaled").show())

+--------+----+---+----+
|unscaled|   0|  1|   2|
+--------+----+---+----+
|       0|NULL|  2|   4|
|       1|  19|  1|NULL|
|       2|  15| 19|   5|
+--------+----+---+----+



---

## 5. Centroids in the original units

A centroid of the scaled model lives in scaled space, where a coordinate of $-0.8$ means nothing
to a reader. Multiplying by the standard deviation and adding back the mean — the two quantities
the fitted scaler is carrying — puts it back into titles and dollars.

In [13]:
fitted_scaler = scaled_pipeline.stages[0]
kmeans_model = scaled_pipeline.stages[1]

mean, std = fitted_scaler.mean, fitted_scaler.std

print(f"{'':<10}" + "".join(f"{c.replace('Cost_Per_Month_', '$'):>16}" for c in COLUMNS))
for i, centre in enumerate(kmeans_model.clusterCenters()):
    original = [centre[j] * std[j] + mean[j] for j in range(len(COLUMNS))]
    print(f"cluster {i:<2}" + "".join(f"{v:>16.1f}" for v in original))

          Total_Library_Size    Num_TV_Shows      Num_Movies          $Basic       $Standard        $Premium
cluster 0           5923.9          3992.5          1931.4             8.9            12.8            16.6
cluster 1           4927.5          3140.5          1787.0             6.6             9.5            12.2
cluster 2           3957.8          2655.1          1302.7            10.6            15.2            20.4


In [14]:
# The inversion is right if it reproduces the per-cluster means of section 4's profile.
import math

exact = {r["prediction"]: [r[c] for c in COLUMNS]
         for r in (scaled_clusters.groupBy("prediction")
                   .agg(*[F.avg(c).alias(c) for c in COLUMNS])
                   .collect())}
for i, centre in enumerate(kmeans_model.clusterCenters()):
    original = [centre[j] * std[j] + mean[j] for j in range(len(COLUMNS))]
    assert all(math.isclose(a, b, rel_tol=1e-6)
               for a, b in zip(original, exact[i]))
print("un-scaled centroids match the cluster means computed from the original columns")

un-scaled centroids match the cluster means computed from the original columns


A centroid table is the output to put in a report. The list of member countries in section 2 is
longer, is unreadable past about twenty names, and does not say what the clusters are *for*; the
centroid table says what each group looks like in units the reader already understands. The member
list is the right output when the question is about a particular country.

---

## What to take away

The six input columns span two scales: catalogue counts in the thousands and prices in single
and double digits. Squared Euclidean distance is dominated by the first three, which contribute
over 99 % of the worst-case distance, so K-means on the raw columns partitions the countries by
catalogue size and ignores what they pay.

A `StandardScaler` before the clustering fixes it, and it is an estimator, so it costs one pass.
The silhouette falls when it is added, because the scaled problem is harder and the two numbers
are computed in different spaces.

**Exercise 7** asks for the parts this notebook leaves out: the silhouette at $k = 2, 3, 4, 5, 6$
in both the unscaled and the scaled feature space, and a judgement about which $k$ to choose. The
sweep is `for k in range(2, 7)` around the two pipelines above.

In [15]:
spark.stop()